# 05 - Hook 机制与 Grad-CAM

## 学习目标

1. 理解 `Tensor.register_hook` 的用法（捕获/修改梯度）
2. 掌握 `Module.register_forward_hook`（提取特征图）
3. 了解 `register_forward_pre_hook` 和 `register_full_backward_hook`
4. 理解 Hook 的执行顺序
5. 用 Hook 实现 Grad-CAM 可视化

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## 1. Tensor.register_hook - 张量级别的钩子

`Tensor.register_hook(fn)` 会在该张量的梯度被计算时调用 `fn(grad)`。

主要用途：
- 捕获非叶子节点的梯度（默认不保留）
- 修改梯度（返回新梯度）

In [ ]:
# 用例 1: 捕获非叶子节点的梯度
x = torch.tensor([2.0], requires_grad=True)
y = x * 3      # y 是非叶子节点
z = y ** 2      # z = 9x^2

# 默认情况下，y 的梯度不会保留
z.backward()
print(f"x.grad: {x.grad}")
print(f"y.grad: {y.grad}  (非叶子节点，梯度不保留)")

In [ ]:
# 使用 register_hook 捕获非叶子节点梯度
x = torch.tensor([2.0], requires_grad=True)
y = x * 3
z = y ** 2

# 用列表保存梯度
y_grad = []
hook = y.register_hook(lambda grad: y_grad.append(grad.clone()))

z.backward()
print(f"x.grad: {x.grad}")
print(f"y 的梯度（通过 hook 捕获）: {y_grad[0]}")
print(f"\n验证: dz/dy = 2y = 2*6 = 12")

# 移除钩子
hook.remove()

In [ ]:
# 用例 2: 修改梯度（梯度裁剪）
x = torch.tensor([5.0], requires_grad=True)
y = x ** 3  # dy/dx = 3x^2 = 75

# 注册 hook: 将梯度裁剪到 [-10, 10]
hook = x.register_hook(lambda grad: torch.clamp(grad, -10, 10))

y.backward()
print(f"原始梯度应为 3*5^2 = 75")
print(f"裁剪后的梯度: {x.grad}  (被裁剪到 10)")

hook.remove()

## 2. Module.register_forward_hook - 提取特征图

```python
hook_fn(module, input, output)
```

在模块的 `forward()` **执行后**调用。最常见用途：提取中间层的特征图。

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, 3, padding=1)
        self.relu1 = nn.ReLU()
        self.conv2 = nn.Conv2d(8, 16, 3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(16, 3)

    def forward(self, x):
        x = self.relu1(self.conv1(x))
        x = self.relu2(self.conv2(x))
        x = self.pool(x).view(x.size(0), -1)
        x = self.fc(x)
        return x


model = SimpleCNN()

# 存储中间特征
feature_maps = {}

def save_feature_hook(name):
    """创建一个 hook 函数，将特征图保存到字典中。"""
    def hook_fn(module, input, output):
        feature_maps[name] = output.detach()
    return hook_fn

# 注册 hook
hooks = []
hooks.append(model.conv1.register_forward_hook(save_feature_hook('conv1')))
hooks.append(model.conv2.register_forward_hook(save_feature_hook('conv2')))

# 前向传播
x = torch.randn(1, 1, 16, 16)
output = model(x)

print("提取的特征图:")
for name, feat in feature_maps.items():
    print(f"  {name}: shape={feat.shape}")

# 移除所有 hook
for h in hooks:
    h.remove()

## 3. register_forward_pre_hook - 前向传播前钩子

```python
hook_fn(module, input)
```

在 `forward()` **执行前**调用。可以用来检查或修改输入。

In [ ]:
model = SimpleCNN()

def pre_hook_fn(module, input):
    """打印每层收到的输入形状。"""
    print(f"  [PRE] {module.__class__.__name__} 接收输入: shape={input[0].shape}")

# 给所有卷积层注册 pre_hook
hooks = []
for name, module in model.named_modules():
    if isinstance(module, nn.Conv2d):
        hooks.append(module.register_forward_pre_hook(pre_hook_fn))

print("前向传播:")
x = torch.randn(1, 1, 16, 16)
output = model(x)

for h in hooks:
    h.remove()

## 4. register_full_backward_hook - 反向传播钩子

```python
hook_fn(module, grad_input, grad_output)
```

在模块的反向传播**完成后**调用。可以用来捕获梯度信息。

In [ ]:
model = SimpleCNN()

grad_info = {}

def backward_hook_fn(name):
    def hook_fn(module, grad_input, grad_output):
        grad_info[name] = {
            'grad_output_shape': grad_output[0].shape if grad_output[0] is not None else None,
            'grad_input_shapes': [g.shape if g is not None else None for g in grad_input],
        }
    return hook_fn

hooks = []
hooks.append(model.conv1.register_full_backward_hook(backward_hook_fn('conv1')))
hooks.append(model.conv2.register_full_backward_hook(backward_hook_fn('conv2')))

# 前向传播 + 反向传播
x = torch.randn(1, 1, 16, 16)
output = model(x)
loss = output.sum()
loss.backward()

print("反向传播梯度信息:")
for name, info in grad_info.items():
    print(f"  {name}:")
    print(f"    grad_output: {info['grad_output_shape']}")
    print(f"    grad_input:  {info['grad_input_shapes']}")

for h in hooks:
    h.remove()

## 5. Hook 执行顺序

让我们验证不同 Hook 的执行时序。

In [ ]:
model = SimpleCNN()
log = []

# 注册三种 hook
hooks = []

def make_pre_hook(name):
    def fn(module, input):
        log.append(f"[1-PRE]     {name}")
    return fn

def make_fwd_hook(name):
    def fn(module, input, output):
        log.append(f"[2-FORWARD] {name}")
    return fn

def make_bwd_hook(name):
    def fn(module, grad_input, grad_output):
        log.append(f"[3-BACKWARD] {name}")
    return fn

for name, module in model.named_modules():
    if isinstance(module, (nn.Conv2d, nn.Linear)):
        hooks.append(module.register_forward_pre_hook(make_pre_hook(name)))
        hooks.append(module.register_forward_hook(make_fwd_hook(name)))
        hooks.append(module.register_full_backward_hook(make_bwd_hook(name)))

# 前向 + 反向
x = torch.randn(1, 1, 16, 16)
output = model(x)
output.sum().backward()

print("Hook 执行顺序:")
for entry in log:
    print(f"  {entry}")

for h in hooks:
    h.remove()

**执行顺序总结：**

```
前向传播: pre_hook -> forward -> forward_hook （按模块顺序）
反向传播: backward_hook                      （按反向顺序）
```

## 6. Grad-CAM 实现

**Grad-CAM**（Gradient-weighted Class Activation Mapping）通过梯度信息来可视化卷积网络"关注"的区域。

### 原理

1. 对目标类别的 score 做反向传播
2. 获取目标卷积层的**特征图**（通过 forward hook）和**梯度**（通过 backward hook）
3. 对梯度做全局平均池化，得到每个通道的权重
4. 用权重对特征图加权求和，再过 ReLU

$$CAM = ReLU\left(\sum_k \alpha_k \cdot A^k\right)$$

其中 $\alpha_k = \frac{1}{Z} \sum_i \sum_j \frac{\partial y^c}{\partial A^k_{ij}}$

In [ ]:
class GradCAM:
    """Grad-CAM 实现。

    Args:
        model: 目标模型。
        target_layer: 要可视化的卷积层。
    """

    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.feature_map = None
        self.gradient = None

        # 注册 hook
        self._hooks = []
        self._hooks.append(
            target_layer.register_forward_hook(self._save_feature_map)
        )
        self._hooks.append(
            target_layer.register_full_backward_hook(self._save_gradient)
        )

    def _save_feature_map(self, module, input, output):
        """保存特征图。"""
        self.feature_map = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        """保存梯度。"""
        self.gradient = grad_output[0].detach()

    def generate(self, input_tensor, target_class=None):
        """生成 Grad-CAM 热力图。

        Args:
            input_tensor: 输入图像张量 [1, C, H, W]。
            target_class: 目标类别索引。None 时使用预测类别。

        Returns:
            cam: 热力图 [H, W]，值域 [0, 1]。
        """
        # 前向传播
        self.model.eval()
        output = self.model(input_tensor)

        # 确定目标类别
        if target_class is None:
            target_class = output.argmax(dim=1).item()

        # 反向传播
        self.model.zero_grad()
        target_score = output[0, target_class]
        target_score.backward()

        # 计算通道权重: 对梯度做全局平均池化
        # gradient shape: [1, C, H, W]
        weights = self.gradient.mean(dim=(2, 3), keepdim=True)  # [1, C, 1, 1]

        # 加权求和
        cam = (weights * self.feature_map).sum(dim=1, keepdim=True)  # [1, 1, H, W]
        cam = F.relu(cam)  # 只保留正贡献
        cam = cam.squeeze()  # [H, W]

        # 归一化到 [0, 1]
        if cam.max() > 0:
            cam = cam / cam.max()

        return cam

    def remove_hooks(self):
        """移除所有钩子。"""
        for h in self._hooks:
            h.remove()

In [ ]:
# 使用 Grad-CAM
model = SimpleCNN()

# 对最后一个卷积层做 Grad-CAM
grad_cam = GradCAM(model, model.conv2)

# 生成随机输入（实际使用时替换为真实图像）
x = torch.randn(1, 1, 16, 16, requires_grad=True)

# 生成 CAM
cam = grad_cam.generate(x, target_class=0)

print(f"输入形状: {x.shape}")
print(f"CAM 形状: {cam.shape}")
print(f"CAM 值域: [{cam.min().item():.4f}, {cam.max().item():.4f}]")
print(f"\n热力图中最大值位置: {torch.where(cam == cam.max())}")

# 清理
grad_cam.remove_hooks()

**Grad-CAM 使用流程：**

```python
# 1. 创建 GradCAM 实例，指定目标层
grad_cam = GradCAM(model, model.layer4[-1].conv2)

# 2. 生成 CAM
cam = grad_cam.generate(image_tensor, target_class=243)

# 3. 将 CAM 上采样到原图大小
cam_resized = F.interpolate(cam.unsqueeze(0).unsqueeze(0), size=(224, 224))

# 4. 叠加到原图上进行可视化
# 5. 用完后清理 hook
grad_cam.remove_hooks()
```

## 7. Hook 使用注意事项

1. **用完要移除**：Hook 会持有模块引用，不移除会导致内存泄漏
2. **使用 `with` 模式管理**（PyTorch 新版支持）：
   ```python
   with model.conv1.register_forward_hook(fn) as hook:
       output = model(x)
   # hook 自动移除
   ```
3. **不要在 hook 中修改输入/输出的就地操作**
4. **`register_backward_hook` 已弃用**，使用 `register_full_backward_hook`

## 8. 练习题

### 练习 1：提取所有卷积层的特征图

编写一个函数，自动为模型中所有 `Conv2d` 层注册 hook 并提取特征图。

In [ ]:
def extract_all_conv_features(model, input_tensor):
    """提取模型中所有 Conv2d 层的特征图。

    Args:
        model: 目标模型。
        input_tensor: 输入张量。

    Returns:
        features: dict，key 为层名，value 为特征图。
    """
    features = {}
    hooks = []
    # TODO: 为所有 Conv2d 注册 forward hook
    # TODO: 前向传播
    # TODO: 移除所有 hook
    return features


# 测试代码
# model = SimpleCNN()
# x = torch.randn(1, 1, 16, 16)
# features = extract_all_conv_features(model, x)
# for name, feat in features.items():
#     print(f"{name}: {feat.shape}")

### 练习 2：梯度监控 Hook

编写一个 hook，监控每层反向传播时梯度的范数，帮助检测梯度爆炸/消失。

In [ ]:
def monitor_gradient_norms(model, input_tensor):
    """监控每层的梯度范数。

    Args:
        model: 目标模型。
        input_tensor: 输入张量。

    Returns:
        grad_norms: dict，key 为层名，value 为梯度范数。
    """
    grad_norms = {}
    hooks = []
    # TODO: 注册 backward hook
    # TODO: 前向传播 + 反向传播
    # TODO: 移除 hook
    return grad_norms


# 测试代码
# model = SimpleCNN()
# x = torch.randn(1, 1, 16, 16)
# norms = monitor_gradient_norms(model, x)
# for name, norm in norms.items():
#     print(f"{name}: grad_norm={norm:.6f}")

## 9. 小结

### Hook 类型总结

| Hook 类型 | 级别 | 触发时机 | 典型用途 |
|----------|------|---------|----------|
| `Tensor.register_hook` | 张量 | 梯度计算时 | 捕获/修改梯度 |
| `register_forward_pre_hook` | 模块 | forward 前 | 检查/修改输入 |
| `register_forward_hook` | 模块 | forward 后 | 提取特征图 |
| `register_full_backward_hook` | 模块 | backward 后 | 捕获梯度信息 |

### Grad-CAM 步骤

1. 用 forward hook 保存目标层的特征图
2. 用 backward hook 保存目标层的梯度
3. 梯度全局平均池化得到通道权重
4. 权重 x 特征图 → ReLU → 归一化

### 注意事项

1. **Hook 用完一定要 `remove()`**
2. 使用 `register_full_backward_hook`（不要用已弃用的 `register_backward_hook`）
3. Hook 中保存的张量要 `.detach()` 或 `.clone()`，避免影响计算图

### 下一步

在下一个 notebook 中，我们将学习 **权重初始化**，了解不同初始化策略对训练的影响。